# IITaku: low-poly to inferred detail
Compact paired SDF experiment. Run the smoke cells before real training. Prepare your meshes on CPU first. No reconstruction quality is promised; compare held-out geometry with baselines. Check your Kaggle GPU quota and save each session output.

In [ ]:
from pathlib import Path
import zipfile, subprocess, sys, os
# Attach the source ZIP as a Kaggle input. Kaggle may unpack it automatically.
source_candidates = list(Path("/kaggle/input").rglob("training/train.py"))
if not source_candidates:
    archives=list(Path("/kaggle/input").rglob("IITaku_Studio_Source*.zip"))
    assert archives, "Attach the supplied source ZIP or unpacked source dataset first."
    with zipfile.ZipFile(archives[0]) as z: z.extractall("/kaggle/working/source")
    source_candidates=list(Path("/kaggle/working/source").rglob("training/train.py"))
ROOT=source_candidates[0].parents[1]
os.chdir(ROOT)
print("Source:", ROOT)
# Editable values for your real prepared-pairs dataset:
PAIRS_ROOT=Path("/kaggle/input/my-reviewed-mesh-pairs/pairs")
OUT=Path("/kaggle/working/iitaku-run")
RESUME=None  # Path to last.pt from an earlier notebook output, if resuming


In [ ]:
subprocess.check_call([sys.executable,"-m","pip","install","numpy>=2,<3","scipy>=1.15,<2","trimesh>=4.9,<5","scikit-image>=0.25,<1"])
import torch
print("Torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## Small smoke test
This synthetic data checks checkpoint/export plumbing. It is not an anime dataset or evidence of useful reconstruction. You can run this cell with GPU disabled.

In [ ]:
SMOKE=Path("/kaggle/working/smoke")
subprocess.check_call([sys.executable,"training/synthetic_pairs.py","--out",str(SMOKE/"pairs"),"--count","24","--size","32"])
subprocess.check_call([sys.executable,"training/train.py","--pairs",str(SMOKE/"pairs"),"--out",str(SMOKE/"run"),"--epochs","2","--width","8","--hours","0.1"])
subprocess.check_call([sys.executable,"training/evaluate.py","--pairs",str(SMOKE/"pairs"),"--checkpoint",str(SMOKE/"run/best.pt"),"--out",str(SMOKE/"evaluation")])


## Real-data preflight
Turn GPU off while preparing meshes on CPU. Attach a private prepared-pairs dataset containing manifest.json and NPZ grids. Confirm licence/provenance, family splits and mesh validity. Activate GPU for the next training cell only after preflight succeeds.

In [ ]:
import json, collections
assert (PAIRS_ROOT/"manifest.json").exists(), "Edit PAIRS_ROOT to your uploaded prepared-pairs dataset."
rows=json.loads((PAIRS_ROOT/"manifest.json").read_text())
print(collections.Counter(r["split"] for r in rows))
assert len({r["object_id"] for r in rows})==len(rows), "Source objects must be unique."
assert all((PAIRS_ROOT/r["file"]).is_file() for r in rows)
assert {"train","val","test"} <= {r["split"] for r in rows}


## Bounded training chunk
Benchmark a short run and memory usage first. The default trains one device; it does not use multi-GPU distribution. Save output after each 3.5-hour chunk. Attach previous output and set RESUME to last.pt for the next chunk. Check remaining quota; the command cannot read your account quota.

In [ ]:
assert torch.cuda.is_available(), "Enable an available Kaggle GPU for real training."
WIDTH=12
if RESUME: WIDTH=torch.load(RESUME,map_location="cpu",weights_only=False)["width"]
cmd=[sys.executable,"training/train.py","--pairs",str(PAIRS_ROOT),"--out",str(OUT),"--hours","3.5","--batch","2","--width",str(WIDTH)]
if RESUME: cmd += ["--resume",str(RESUME)]
subprocess.check_call(cmd)


## Held-out evaluation
Inspect exported OBJ surfaces in Blender. Report identity/subdivision/refiner metrics. Do not merge validation objects into the test set or release solely on training loss.

In [ ]:
subprocess.check_call([sys.executable,"training/evaluate.py","--pairs",str(PAIRS_ROOT),"--checkpoint",str(OUT/"best.pt"),"--out",str(OUT/"evaluation")])
print("Save notebook output containing checkpoints, metrics and predicted OBJ files.")
